In [1]:
import os
import json
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# =====================================================================
# ⚙️ CONFIGURATION (CẤU HÌNH ĐƯỜNG DẪN - DỄ DÀNG CHỈNH SỬA)
# =====================================================================
MODEL_HPA_PATH = "/kaggle/input/datasets/habao2603/rukopys-trocr-hpa-branch-finetuned-v2-3-phase/final_cyrillic_htr_model/final_cyrillic_htr_model"  # Đường dẫn tới thư mục checkpoint model HPA mới (TrOCR)
IMAGE_DIR = "/kaggle/input/datasets/habao2603/rukopys-test-only/images"                                   # Thư mục chứa các ảnh test (.jpg/.png) gốc
OLD_SUBMISSION_PATH = "/kaggle/input/datasets/habao2603/tdttfirst-entry-submission/submission (2).csv"                 # Đường dẫn file submission cũ cần thay thế text (Ví dụ: "submission (2).csv")
NEW_SUBMISSION_PATH = "/kaggle/working/submission.csv"             # Đường dẫn xuất file kết quả mới để nộp lên Kaggle

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HPA_TYPES = {"handwritten", "printed", "annotation"}

print(f"🚀 Khởi tạo pipeline nâng cấp OCR (V2). Thiết bị sử dụng: {DEVICE.upper()}")

# =====================================================================
# 🤖 STEP 1: TẢI MÔ HÌNH HPA MỚI VÀ KIỂM TRA VOCABULARY
# =====================================================================
print(f"🔄 Đang nạp TrOCR Processor và Model từ thư mục: {MODEL_HPA_PATH}...")
try:
    processor = TrOCRProcessor.from_pretrained(MODEL_HPA_PATH)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_HPA_PATH).to(DEVICE)
    model.eval()
    print("✅ Tải mô hình thành công!")
    
    # Kiểm tra kích thước từ vựng (Vocab Size) để đảm bảo từ vựng mới đã được nhận diện
    vocab_size_tokenizer = len(processor.tokenizer)
    vocab_size_model = model.config.decoder.vocab_size
    print(f"📊 [VOCAB CHECK] Kích thước từ vựng trong Tokenizer: {vocab_size_tokenizer}")
    print(f"📊 [VOCAB CHECK] Kích thước từ vựng trong Model Decoder: {vocab_size_model}")
    if vocab_size_tokenizer != vocab_size_model:
        print(f"⚠️ [WARNING] Cảnh báo: Kích thước vocab không đồng bộ giữa Tokenizer và Model!")
    else:
        print("🎉 [OK] Hệ thống từ vựng (Vocab) hoàn toàn khớp và đồng bộ!")
except Exception as e:
    print(f"❌ [CRITICAL ERROR] Không thể tải model. Vui lòng kiểm tra lại đường dẫn MODEL_HPA_PATH. Chi tiết: {e}")
    raise e

# =====================================================================
# 🧠 STEP 2: THỰC THI PIPELINE INFERENCE VÀ CẬP NHẬT CỤC BỘ TEXT
# =====================================================================
if not os.path.exists(OLD_SUBMISSION_PATH):
    raise FileNotFoundError(f"❌ Không tìm thấy file submission cũ tại: '{OLD_SUBMISSION_PATH}'")

df_sub = pd.read_csv(OLD_SUBMISSION_PATH)

# Đếm tổng số lượng region HPA có sẵn trong file submission cũ để làm log đối chiếu
total_hpa_regions_found = 0
for idx, row in df_sub.iterrows():
    regions = json.loads(row['regions'])
    for r in regions:
        if r.get('type') in HPA_TYPES:
            total_hpa_regions_found += 1

print(f"🔍 [LOG] Tìm thấy tổng cộng {total_hpa_regions_found} vùng thuộc nhánh HPA trong file submission cũ.")
print("⚡ Bắt đầu quét qua từng ảnh và tiến hành thay thế kết quả OCR bằng model mới...")

updated_regions_count = 0
samples_log = []

for idx, row in tqdm(df_sub.iterrows(), total=len(df_sub), desc="Inference HPA"):
    img_name = row['image']
    img_path = os.path.join(IMAGE_DIR, img_name)
    
    # Thống kê cảnh báo nếu thiếu file ảnh trên đĩa cứng Kaggle
    if not os.path.exists(img_path):
        print(f"⚠️ [WARNING] Thiếu ảnh: không tìm thấy file '{img_path}'. Bỏ qua cập nhật cho ảnh này.")
        continue
        
    try:
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
    except Exception as e:
        print(f"❌ [ERROR] Không thể mở file ảnh {img_name}: {e}")
        continue
        
    regions = json.loads(row['regions'])
    regions_changed = False
    
    for region in regions:
        r_type = region.get('type')
        
        # Chỉ xử lý các vùng thuộc handwritten, printed, hoặc annotation
        if r_type in HPA_TYPES:
            bbox = region.get('bbox')  # Định dạng [x1, y1, x2, y2] lấy trực tiếp trong file cũ
            if not bbox or len(bbox) != 4:
                continue
                
            x1, y1, x2, y2 = bbox
            # Cắt gọn biên tọa độ (Clip coordinates) để chống tràn viền ảnh
            x1, x2 = max(0, min(int(x1), w)), max(0, min(int(x2), w))
            y1, y2 = max(0, min(int(y1), h)), max(0, min(int(y2), h))
            
            if x2 <= x1 or y2 <= y1:
                print(f"⚠️ [WARNING] Tọa độ Bbox đảo ngược hoặc không hợp lệ {bbox} trên ảnh {img_name}.")
                continue
                
            # Cắt vùng ảnh (Crop) theo đúng tọa độ bbox lưu trong submission
            cropped_img = img.crop((x1, y1, x2, y2))
            
            # Đưa qua mô hình TrOCR mới để lấy chuỗi ký tự kết quả
            try:
                pixel_values = processor(images=cropped_img, return_tensors="pt").pixel_values.to(DEVICE)
                with torch.no_grad():
                    generated_ids = model.generate(pixel_values, max_length=128)
                pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            except Exception as e:
                print(f"❌ [ERROR] Lỗi khi nhận diện chữ tại bbox {bbox} của ảnh {img_name}: {e}")
                pred_text = region.get('text', '')  # Nếu lỗi, giữ nguyên text cũ để an toàn điểm số
                
            old_text = region.get('text', '')
            
            # Ghi lại một vài mẫu để hiển thị log kiểm tra trước/sau
            if len(samples_log) < 5 and old_text != pred_text:
                samples_log.append({
                    "image": img_name,
                    "type": r_type,
                    "bbox": bbox,
                    "old": old_text,
                    "new": pred_text
                })
                
            # Cập nhật ghi đè text mới vào cấu trúc vùng
            region['text'] = pred_text
            updated_regions_count += 1
            regions_changed = True

    # Lưu lại chuỗi JSON mới vào đúng vị trí dòng hiện tại nếu có thay đổi chữ
    if regions_changed:
        df_sub.at[idx, 'regions'] = json.dumps(regions, ensure_ascii=False)

# =====================================================================
# 💾 STEP 3: XUẤT FILE SUBMISSION MỚI
# =====================================================================
df_sub.to_csv(NEW_SUBMISSION_PATH, index=False)

print("\n" + "="*60)
print("🏁 TIẾN TRÌNH THAY THẾ OCR THÀNH CÔNG VÀ HOÀN TẤT!")
print(f"📊 [LOG] Tổng số vùng HPA tìm thấy ban đầu: {total_hpa_regions_found}")
print(f"📊 [LOG] Tổng số vùng chữ đã được mô hình mới cập nhật/thay thế: {updated_regions_count}")
print(f"💾 File submission mới sẵn sàng nộp Kaggle tại: {NEW_SUBMISSION_PATH}")
print("="*60)

# In các mẫu text trước/sau thay đổi để đánh giá nhanh chất lượng mô hình mới
if samples_log:
    print("\n📝 [SAMPLES LOG] Xem trước một số vùng thay đổi OCR sau khi áp dụng mô hình mới:")
    for i, sample in enumerate(samples_log, 1):
        print(f"\n🔹 Mẫu {i} | Ảnh: {sample['image']} | Thể loại: {sample['type'].upper()} | Bbox: {sample['bbox']}")
        print(f"   [-] Văn bản cũ (Model cũ):  '{sample['old']}'")
        print(f"   [+] Văn bản mới (Model mới): '{sample['new']}'")
else:
    print("\nℹ️ [LOG] Lưu ý: Không phát hiện sự khác biệt nào về chuỗi văn bản trong các mẫu đầu tiên.")

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


🚀 Khởi tạo pipeline nâng cấp OCR (V2). Thiết bị sử dụng: CUDA
🔄 Đang nạp TrOCR Processor và Model từ thư mục: /kaggle/input/datasets/habao2603/rukopys-trocr-hpa-branch-finetuned-v2-3-phase/final_cyrillic_htr_model/final_cyrillic_htr_model...


Loading weights:   0%|          | 0/636 [00:00<?, ?it/s]

✅ Tải mô hình thành công!
📊 [VOCAB CHECK] Kích thước từ vựng trong Tokenizer: 50342
📊 [VOCAB CHECK] Kích thước từ vựng trong Model Decoder: 50342
🎉 [OK] Hệ thống từ vựng (Vocab) hoàn toàn khớp và đồng bộ!
🔍 [LOG] Tìm thấy tổng cộng 7343 vùng thuộc nhánh HPA trong file submission cũ.
⚡ Bắt đầu quét qua từng ảnh và tiến hành thay thế kết quả OCR bằng model mới...


Inference HPA: 100%|██████████| 385/385 [1:03:25<00:00,  9.88s/it]


🏁 TIẾN TRÌNH THAY THẾ OCR THÀNH CÔNG VÀ HOÀN TẤT!
📊 [LOG] Tổng số vùng HPA tìm thấy ban đầu: 7343
📊 [LOG] Tổng số vùng chữ đã được mô hình mới cập nhật/thay thế: 7343
💾 File submission mới sẵn sàng nộp Kaggle tại: /kaggle/working/submission.csv

📝 [SAMPLES LOG] Xem trước một số vùng thay đổi OCR sau khi áp dụng mô hình mới:

🔹 Mẫu 1 | Ảnh: 33331ef7-02fe-4d53-a1c2-19940d128b49.jpg | Thể loại: HANDWRITTEN | Bbox: [807, 366, 2409, 479]
   [-] Văn bản cũ (Model cũ):  'Жити треба цікаво: читати книжки, ходити в театр,'
   [+] Văn bản mới (Model mới): 'Жити треба цінаво: читати книжки, ходити в театр,'

🔹 Mẫu 2 | Ảnh: 33331ef7-02fe-4d53-a1c2-19940d128b49.jpg | Thể loại: HANDWRITTEN | Bbox: [792, 494, 1559, 582]
   [-] Văn bản cũ (Model cũ):  'тренерувати пса Патрона.'
   [+] Văn bản mới (Model mới): 'тренерувати жа Патрона.'

🔹 Mẫu 3 | Ảnh: 33331ef7-02fe-4d53-a1c2-19940d128b49.jpg | Thể loại: HANDWRITTEN | Bbox: [851, 585, 2432, 697]
   [-] Văn bản cũ (Model cũ):  'Треба жити! Не впівсими, 